In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/superstore.csv", encoding="latin1")
df.shape

(9994, 21)

In [2]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
df.isnull().sum()

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

## Clean and prepare data

In [4]:
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%m/%d/%Y")
df = df.dropna(subset=["Sales"])

In [5]:
daily_sales = df.groupby("Order Date")["Sales"].sum().reset_index()
daily_sales = daily_sales.sort_values("Order Date")
daily_sales = daily_sales.set_index("Order Date").asfreq("D", fill_value=0).reset_index()
daily_sales.head()

,Order Date,Sales
0,2014-01-03,16.448
1,2014-01-04,288.060
2,2014-01-05,19.536
3,2014-01-06,4407.100
4,2014-01-07,87.158


## Feature engineering

In [6]:
daily_sales["year"] = daily_sales["Order Date"].dt.year
daily_sales["month"] = daily_sales["Order Date"].dt.month
daily_sales["day"] = daily_sales["Order Date"].dt.day
daily_sales["day_of_week"] = daily_sales["Order Date"].dt.dayofweek
daily_sales["quarter"] = daily_sales["Order Date"].dt.quarter

In [7]:
daily_sales["sales_lag1"] = daily_sales["Sales"].shift(1)
daily_sales["rolling_mean_7"] = daily_sales["Sales"].rolling(window=7).mean()

In [8]:
daily_sales = daily_sales.dropna().reset_index(drop=True)
daily_sales.shape

(1452, 9)

In [9]:
daily_sales.head()

,Order Date,Sales,year,month,day,day_of_week,quarter,sales_lag1,rolling_mean_7
0,2014-01-09,40.544,2014,1,9,3,1,0.000,694.120857
1,2014-01-10,54.830,2014,1,10,4,1,40.544,699.604000
2,2014-01-11,9.940,2014,1,11,5,1,54.830,659.872571
3,2014-01-12,0.000,2014,1,12,6,1,9.940,657.081714
4,2014-01-13,3553.795,2014,1,13,0,1,0.000,535.181000


## Save cleaned data

In [10]:
daily_sales.to_csv("outputs/cleaned_data.csv", index=False)